# Tinh chỉnh một phần RAD-DINO — CXR Nodule Detection

Notebook này chạy **mục 8.4** trong `CLAUDE.md`: mở băng vài tầng cuối của
RAD-DINO. Đây là hạng mục duy nhất bị chặn bởi 6GB VRAM của RTX 3060 ở nhà.

**Nó trả lời câu hỏi gì?**

Bài báo hiện nói: tăng cường dữ liệu làm *giảm* 20,5 điểm, *vì* backbone đóng
băng nên không thích nghi được với ảnh đã biến đổi. Đó mới là **giả thuyết**,
chưa ai kiểm chứng. Notebook chạy hai nhánh:

| Nhánh | Mở băng | Tăng cường | Ý nghĩa |
|---|---|---|---|
| A | 4 tầng cuối | không | mốc so sánh |
| B | 4 tầng cuối | có (8 biến thể) | **nhánh quyết định** |

- Nếu B ≳ A: giả thuyết **đúng**, tăng cường hết phản tác dụng khi backbone học được.
- Nếu B vẫn ≪ A: lời giải thích trong bài **sai**, phải viết lại phần thảo luận.

Cả hai kết quả đều dùng được cho bài báo. Kết quả "sai" thậm chí còn đáng giá hơn.

> **Bật GPU trước khi chạy:** Runtime → Change runtime type → A100 hoặc L4.

## 1. Kiểm tra GPU

Dừng lại nếu ô này báo không có GPU — mọi thứ phía sau sẽ chạy trên CPU và
mất hàng chục giờ.

In [ ]:
import torch, subprocess
print(subprocess.run(["nvidia-smi","--query-gpu=name,memory.total",
                      "--format=csv,noheader"],capture_output=True,text=True).stdout.strip())
assert torch.cuda.is_available(), "CHUA BAT GPU: Runtime -> Change runtime type -> A100/L4"
gb = torch.cuda.get_device_properties(0).total_memory/1e9
print(f"torch {torch.__version__} | {torch.cuda.get_device_name(0)} | {gb:.0f} GB")
print("Batch de nghi:", 8 if gb > 30 else (4 if gb > 20 else 2))

## 2. Lấy mã nguồn từ GitHub

Điền `REPO` cho khớp repo riêng của bạn. Token lưu trong Colab Secrets
(biểu tượng chìa khoá ở thanh bên trái), **đừng dán thẳng vào ô code** —
notebook có thể bị chia sẻ kèm token.

Chạy lại đúng ô này mỗi khi mình sửa code trên máy: nó `git pull`, không clone lại.

In [ ]:
import os
REPO = "bvxia/cxr-nodule"          # <-- SUA cho khop repo cua ban
BRANCH = "main"

from google.colab import userdata
try:
    TOKEN = userdata.get("GH_TOKEN")     # Colab Secrets -> ten GH_TOKEN
except Exception:
    from getpass import getpass
    TOKEN = getpass("GitHub token: ")

url = f"https://{TOKEN}@github.com/{REPO}.git"
if os.path.isdir("/content/cxr/.git"):
    !cd /content/cxr && git fetch -q origin && git reset -q --hard origin/{BRANCH}
else:
    !git clone -q -b {BRANCH} {url} /content/cxr
%cd /content/cxr
!git log --oneline -3

## 3. Cài thư viện

`transformers` để nạp RAD-DINO. Colab đã có torch/opencv/scipy sẵn.

In [ ]:
!pip install -q transformers==4.44.2 pydicom
import transformers; print("transformers", transformers.__version__)

## 4. Dữ liệu

Tải `cxr_colab_data.tar.gz` (83 MB) lên Drive **một lần duy nhất**, rồi ô này
giải nén vào `/content` cho nhanh (đọc thẳng từ Drive chậm hơn nhiều).

Gói chứa: 247 ảnh JSRT đã xử lý, 247 ảnh đã khử xương, `labels_processed.csv`,
`transforms.json`, và U-Net khử xương.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

TARBALL = "/content/drive/MyDrive/cxr_colab_data.tar.gz"   # <-- SUA neu de cho khac
assert os.path.exists(TARBALL), f"Khong thay {TARBALL} — hay tai len Drive truoc"

!mkdir -p /content/cxr_data && tar xzf {TARBALL} -C /content/cxr_data --strip-components=0
!ls /content/cxr_data/data/jsrt/processed
# script mong doi /content/cxr_data/jsrt/... nen tao lien ket cho khop config
!mkdir -p /content/cxr_data/checkpoints
!ln -sfn /content/cxr_data/data/jsrt /content/cxr_data/jsrt
n_img = len(os.listdir("/content/cxr_data/jsrt/processed/images"))
n_bs  = len(os.listdir("/content/cxr_data/jsrt/processed/images_bs"))
print(f"anh: {n_img} | anh khu xuong: {n_bs}")
assert n_img == 247 and n_bs == 247, "thieu anh — giai nen loi?" 

## 5. Kiểm tra trước khi chạy thật

`--check` nạp mô hình + dữ liệu, chạy một lượt thuận rồi thoát. Nếu ô này
chạy được thì hai nhánh phía dưới sẽ không chết vì lỗi cấu hình.

In [ ]:
!cd /content/cxr && python raddino_finetune.py \
    --config configs/jsrt_colab_ft.yaml --unfreeze 4 --check

## 6. Nhánh A — tinh chỉnh, KHÔNG tăng cường

10 fold, mỗi fold dựng lại backbone từ trọng số gốc (bắt buộc: dùng lại
backbone đã tinh chỉnh của fold trước là **rò rỉ dữ liệu kiểm tra**).

Ước tính trên A100: khoảng 40–70 phút. Giảm `--epochs` nếu muốn thử nhanh.

In [ ]:
!cd /content/cxr && python raddino_finetune.py \
    --config configs/jsrt_colab_ft.yaml \
    --unfreeze 4 --variants 1 --epochs 30 --batch 4 \
    --tag ft4_noaug 2>&1 | tail -40

## 7. Nhánh B — tinh chỉnh, CÓ tăng cường

**Đây là nhánh quyết định.** 8 biến thể mỗi ảnh nên lâu hơn nhánh A khoảng
tám lần; cân nhắc giảm `--epochs` xuống 10–15 để tổng thời gian tương đương.

In [ ]:
!cd /content/cxr && python raddino_finetune.py \
    --config configs/jsrt_colab_ft.yaml \
    --unfreeze 4 --variants 8 --epochs 15 --batch 4 \
    --tag ft4_aug 2>&1 | tail -40

## 8. Khoảng tin cậy và so sánh

`bootstrap_ci.py` lấy mẫu lại **theo ảnh** và tính lại ngưỡng trong từng lần
lấy mẫu. Chế độ `--compare` bắt cặp trên cùng tập ảnh, nên nó đặt khoảng tin
cậy lên chính **hiệu số** — đó mới là thứ trả lời được câu hỏi ở đầu notebook.

In [ ]:
!cd /content/cxr && python bootstrap_ci.py runs/scores_ft4_noaug.json
!cd /content/cxr && python bootstrap_ci.py runs/scores_ft4_aug.json

In [ ]:
# Hieu so B - A. Neu khoang tin cay chua 0 thi KHONG ket luan duoc gi.
!cd /content/cxr && python bootstrap_ci.py --compare \
    runs/scores_ft4_noaug.json runs/scores_ft4_aug.json

## 9. Lưu kết quả về Drive

Colab xoá `/content` khi ngắt phiên. Ô này chép `runs/` về Drive để mình đọc
lại ở phiên sau.

In [ ]:
!mkdir -p /content/drive/MyDrive/cxr_runs
!cp -v /content/cxr/runs/*.json /content/drive/MyDrive/cxr_runs/ 2>/dev/null | tail -20
print("\nDa luu. Bao mình biet, mình se doc lai tu Drive hoac ban tai ve.")

## Diễn giải — đọc trước khi kết luận

1. **Khoảng tin cậy chồng lấn = không kết luận được.** Với 154 nốt, chênh
   lệch vài điểm CPM gần như chắc chắn nằm trong nhiễu.
2. **Nhóm subtlety 1 chỉ có 25 nốt, nhóm 5 có 12.** Một nốt đổi trạng thái
   là 4% ở nhóm 1 và 8,3% ở nhóm 5. `CLAUDE.md` mục 4 đã cảnh báo đúng
   chuyện này: chênh lệch 8,0% với 4,0% chỉ là **một nốt**.
3. **Các chỉ số phân tầng chưa hiệu chỉnh đa so sánh** — coi là thăm dò,
   đừng đặt chúng làm luận điểm chính của bài.
4. **Đừng bật `--amp`.** `CLAUDE.md` mục 5 ghi AMP từng làm loss bằng đúng 0.